## 1. SharedResponses.csv

In [1]:
import pandas as pd
from pathlib import Path

import os


In [2]:
filter_cols = ["ScenarioType", "DiffNumberOFCharacters", "DescriptionShown",
               "CrossingSignal", "Dog", "Cat", "Stroller", "Homeless", "Criminal",
               "DefaultChoice", "NonDefaultChoice"]

parts = []
for chunk in pd.read_csv("raw_data/SharedResponses.csv", chunksize=1_000_000):
    c = chunk[(chunk["ScenarioType"] == "Gender") &
              (chunk["DiffNumberOFCharacters"] == 0) &
              (chunk["CrossingSignal"] == 0) &
              (chunk["Dog"] == 0) &
              (chunk["Cat"] == 0) &
              (chunk["Stroller"] == 0) &
              (chunk["Homeless"] == 0) &
              (chunk["Criminal"] == 0) &
              (chunk["Pregnant"] == 0)]
    parts.append(c.drop(columns=filter_cols))
    del chunk, c

gender_df = pd.concat(parts, ignore_index=True)
del parts  # load as chunks to avoid memory issues

/var/folders/7x/9fs4vb8n44s4zw83h0qv_6hm0000gn/T/ipykernel_77848/47841108.py:6: DtypeWarning: Columns (0: Man) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv("raw_data/SharedResponses.csv", chunksize=1_000_000):
/var/folders/7x/9fs4vb8n44s4zw83h0qv_6hm0000gn/T/ipykernel_77848/47841108.py:6: DtypeWarning: Columns (0: Man) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv("raw_data/SharedResponses.csv", chunksize=1_000_000):
/var/folders/7x/9fs4vb8n44s4zw83h0qv_6hm0000gn/T/ipykernel_77848/47841108.py:6: DtypeWarning: Columns (0: Man) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv("raw_data/SharedResponses.csv", chunksize=1_000_000):
/var/folders/7x/9fs4vb8n44s4zw83h0qv_6hm0000gn/T/ipykernel_77848/47841108.py:6: DtypeWarning: Columns (0: Man) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd

In [3]:
keys = ["ResponseID", "ExtendedSessionID", "UserID"]
pair_size = gender_df.groupby(keys)["ResponseID"].transform("size")
gender_df = gender_df[pair_size == 2].reset_index(drop=True)

print(f"complete pairs: {len(gender_df)/2:,}")

complete pairs: 2,871,378.0


In [5]:
example = gender_df[gender_df["ResponseID"] == "222eXyoNew3yhozFZ"]
example

,ResponseID,ExtendedSessionID,UserID,ScenarioOrder,Intervention,PedPed,Barrier,AttributeLevel,ScenarioTypeStrict,DefaultChoiceIsOmission,...,Boy,Girl,LargeWoman,LargeMan,MaleExecutive,FemaleExecutive,FemaleAthlete,MaleAthlete,FemaleDoctor,MaleDoctor
4,222eXyoNew3yhozFZ,-2143395746_5897031988670582.0,5.897032e+15,9,0,1,0,Female,Gender,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
724625,222eXyoNew3yhozFZ,-2143395746_5897031988670582.0,5.897032e+15,9,1,1,0,Male,Gender,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


## 2. SharedResponseSurvey.csv

In [6]:
keep_cols = ["ExtendedSessionID", "Review_age", "Review_education", "Review_gender",
             "Review_income", "Review_political", "Review_religious"]

# only load the columns we use (much less memory)
survey_df = pd.read_csv("raw_data/SharedResponsesSurvey.csv", usecols=keep_cols,
                        dtype={"Review_age": "str"})

survey_df.head(5)


,ExtendedSessionID,Review_age,Review_education,Review_gender,Review_income,Review_political,Review_religious
0,-2147481563_6968366610050747.0,32,bachelor,male,35000,0.5,0.61
1,-2147481563_6968366610050747.0,32,bachelor,male,35000,0.5,0.61
2,-2147481563_6968366610050747.0,32,bachelor,male,35000,0.5,0.61
3,-2147481563_6968366610050747.0,32,bachelor,male,35000,0.5,0.61
4,-2147481563_6968366610050747.0,32,bachelor,male,35000,0.5,0.61


Merge survey dataset to session dataset through ExtendedSessionID

In [7]:
survey = survey_df
del survey_df

# error 
survey["Review_age"] = pd.to_numeric(survey["Review_age"], errors="coerce")

survey = survey.drop_duplicates()
print(survey["ExtendedSessionID"].duplicated().sum()) 

survey.head(5)


0


,ExtendedSessionID,Review_age,Review_education,Review_gender,Review_income,Review_political,Review_religious
0,-2147481563_6968366610050747.0,32.0,bachelor,male,35000,0.50,0.61
12,-2147456672_5603843098383972.0,23.0,graduate,male,35000,0.65,0.00
22,-2147455629_9512031443725464.0,15.0,others,male,default,0.68,0.00
34,-2147423715_955468113069397.0,15.0,high,male,under5000,0.81,0.00
45,-2147422537_8735659664189848.0,23.0,bachelor,male,50000,1.00,0.00


In [8]:
df = gender_df.merge(
    survey,
    on="ExtendedSessionID",
    how="inner",    
    validate="many_to_one",  
)


In [9]:
df.head(5)

,ResponseID,ExtendedSessionID,UserID,ScenarioOrder,Intervention,PedPed,Barrier,AttributeLevel,ScenarioTypeStrict,DefaultChoiceIsOmission,...,FemaleAthlete,MaleAthlete,FemaleDoctor,MaleDoctor,Review_age,Review_education,Review_gender,Review_income,Review_political,Review_religious
0,222eXyoNew3yhozFZ,-2143395746_5897031988670582.0,5.897032e+15,9,0,1,0,Female,Gender,0.0,...,1.0,0.0,0.0,0.0,NaN,underHigh,male,under5000,1.00,0.00
1,222wBcHAXk6HekLYd,-662808727_9558009616420568.0,9.558010e+15,3,0,1,0,Female,Gender,0.0,...,0.0,0.0,1.0,0.0,28.0,bachelor,male,25000,0.50,0.65
2,223Ft2wHTr3vzyvtK,2007775256_9435757699885670.0,9.435758e+15,3,0,0,0,Male,Gender,1.0,...,0.0,0.0,0.0,1.0,2000.0,others,others,above100000,0.00,1.00
3,223gkKGNvSHRsvibk,1870568321_3707873580228009.0,3.707874e+15,7,0,0,0,Female,Gender,0.0,...,0.0,0.0,0.0,0.0,31.0,graduate,male,10000,0.17,0.81
4,224Qeqo68YbTQPR9t,-640500409_654418766053563.0,6.544188e+14,7,0,1,0,Male,Gender,1.0,...,0.0,0.0,0.0,0.0,19.0,high,male,under5000,0.50,0.19


In [10]:
example = df[df["ResponseID"] == "222eXyoNew3yhozFZ"]
print(example.iloc[0])
print(example.iloc[1])

ResponseID                              222eXyoNew3yhozFZ
ExtendedSessionID          -2143395746_5897031988670582.0
UserID                                 5897031988670580.0
ScenarioOrder                                           9
Intervention                                            0
PedPed                                                  1
Barrier                                                 0
AttributeLevel                                     Female
ScenarioTypeStrict                                 Gender
DefaultChoiceIsOmission                               0.0
NumberOfCharacters                                    3.0
Saved                                                   1
Template                                          Desktop
LeftHand                                              1.0
UserCountry3                                          GBR
Man                                                   0.0
Woman                                                 0.0
Pregnant      

In [11]:
pairs = {
    "Woman": "Man",
    "OldWoman": "OldMan",
    "Girl": "Boy",
    "LargeWoman": "LargeMan",
    "FemaleExecutive": "MaleExecutive",
    "FemaleAthlete": "MaleAthlete",
    "FemaleDoctor": "MaleDoctor",
}
female_cols = list(pairs.keys())
male_cols = list(pairs.values())
keys = ["ResponseID", "ExtendedSessionID", "UserID"]

# all these cols should be numeric 
for col in female_cols + male_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

female = df[df["AttributeLevel"] == "Female"].set_index(keys)
male = df[df["AttributeLevel"] == "Male"].set_index(keys)

# keep only sessions that have both a Female and a Male row
both = female.index.intersection(male.index)
female = female.loc[both]
male = male.loc[both]

# assuming every scenario is valid
valid = pd.Series(True, index=both)

# each female type matches its male counterpart
for f_col, m_col in pairs.items():
    same = female[f_col] == male[m_col]
    print(f"{f_col} vs {m_col}: {(~same).sum():,} mismatches")
    valid = valid & same

# no men on the Female side, no women on the Male side
no_men = female[male_cols].sum(axis=1) == 0
no_women = male[female_cols].sum(axis=1) == 0
print(f"men on Female side: {(~no_men).sum():,}")
print(f"women on Male side: {(~no_women).sum():,}")
valid = valid & no_men & no_women

# keep only valid scenarios (both rows) in df
valid_ids = valid[valid].index
df = df[df.set_index(keys).index.isin(valid_ids)].reset_index(drop=True)


Woman vs Man: 0 mismatches
OldWoman vs OldMan: 0 mismatches
Girl vs Boy: 0 mismatches
LargeWoman vs LargeMan: 0 mismatches
FemaleExecutive vs MaleExecutive: 0 mismatches
FemaleAthlete vs MaleAthlete: 0 mismatches
FemaleDoctor vs MaleDoctor: 0 mismatches
men on Female side: 0
women on Male side: 0


In [12]:
df = df[df["AttributeLevel"] == "Female"]

print(f"{len(df):,}")

459,643


In [13]:
clusters = pd.read_csv("raw_data/country_cluster_map.csv")
clusters["Cluster"] = clusters["Cluster"].map({0: "Western", 1: "Eastern", 2: "Southern"})

df = df.merge(
    clusters[["ISO3", "Cluster"]],
    left_on="UserCountry3", right_on="ISO3",
    how="inner",
    validate="many_to_one",
)

df.head(5)

,ResponseID,ExtendedSessionID,UserID,ScenarioOrder,Intervention,PedPed,Barrier,AttributeLevel,ScenarioTypeStrict,DefaultChoiceIsOmission,...,FemaleDoctor,MaleDoctor,Review_age,Review_education,Review_gender,Review_income,Review_political,Review_religious,ISO3,Cluster
0,222eXyoNew3yhozFZ,-2143395746_5897031988670582.0,5.897032e+15,9,0,1,0,Female,Gender,0.0,...,0.0,0.0,NaN,underHigh,male,under5000,1.00,0.00,GBR,Western
1,222wBcHAXk6HekLYd,-662808727_9558009616420568.0,9.558010e+15,3,0,1,0,Female,Gender,0.0,...,1.0,0.0,28.0,bachelor,male,25000,0.50,0.65,CHL,Southern
2,223gkKGNvSHRsvibk,1870568321_3707873580228009.0,3.707874e+15,7,0,0,0,Female,Gender,0.0,...,0.0,0.0,31.0,graduate,male,10000,0.17,0.81,HUN,Southern
3,224qjZne8dMpkTTFP,931998731_9471183296199690.0,9.471183e+15,12,0,1,0,Female,Gender,0.0,...,0.0,0.0,29.0,bachelor,male,80000,0.39,1.00,USA,Western
4,224vHgRAsafmKMyuF,336950527_3158078381009448.0,3.158078e+15,4,0,0,0,Female,Gender,0.0,...,0.0,0.0,31.0,graduate,male,10000,0.84,0.01,ITA,Western


In [14]:
df = df[df["Cluster"] == "Western"]

print(f"{len(df):,}")

346,549


In [15]:
os.makedirs("extracted_data", exist_ok=True)
df.to_csv("extracted_data/dataset.csv", index=False)
